In [80]:
# ============================================================
# STEP 1
# Load the dataset
# ============================================================

import pandas as pd

class WineDataLoader:

    def __init__(self, file_path):
        # Store the file path so the object "remembers" where the dataset is.
        self.file_path = file_path

    def load_data(self):
        # Load the CSV file using pandas.
        # The caller does not need to know how pandas.read_csv works.
        wine_dataset = pd.read_csv(self.file_path)
        return wine_dataset



loader = WineDataLoader("../data/raw/wine.csv")  # Create the loader object
wine_dataset = loader.load_data()                # Load the dataset

print(wine_dataset.head())  # Show first rows


   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8        5  
2      9.8        5 

In [81]:
# ============================================================
# STEP 2
# Preprocess the dataset
# ============================================================

from sklearn.model_selection import train_test_split

class WinePreprocessor:

    def __init__(self, test_size: float = 0.2, random_state: int = 42):
        # Store configuration values for splitting.
        self.test_size = test_size
        self.random_state = random_state

    def create_features(self, wine_dataset):
        """
        Extract all columns EXCEPT 'quality'.
        This becomes X (features).
        """
        X = wine_dataset.drop("quality", axis=1)
        return X

    def create_target(self, wine_dataset):
        """
        Convert wine quality into binary labels:
            - 1 = good wine (quality >= 7)
            - 0 = bad wine
        """
        y = wine_dataset["quality"].apply(
            lambda quality: 1 if quality >= 7 else 0
        )
        return y

    def split_data(self, X, y):
        """
        Split the dataset into training and testing sets.
        """
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=self.test_size,
            random_state=self.random_state
        )
        return X_train, X_test, y_train, y_test


# ============================================================
# USING THE PREPROCESSOR (Composition)
# ============================================================

preprocessor = WinePreprocessor()

# Create features and target
X = preprocessor.create_features(wine_dataset)
y = preprocessor.create_target(wine_dataset)

# Split the data
X_train, X_test, y_train, y_test = preprocessor.split_data(X, y)

# ============================================================
# DISPLAY RESULTS
# ============================================================

print("Original dataset:", wine_dataset.shape)
print("Features:", X.shape)
print("Target:", y.shape)
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


Original dataset: (1599, 12)
Features: (1599, 11)
Target: (1599,)
Training features: (1279, 11)
Testing features: (320, 11)
Training target: (1279,)
Testing target: (320,)


                  BaseWineModel
                  🔵 Abstraction
                       │
             ┌─────────┴─────────┐
             │                   │
             ↓                   ↓
 RandomForestWineModel   DecisionTreeWineModel
        🟢                    🟢
  Inheritance              Inheritance
        │                       │
        ↓                       ↓
    train()                 train()
    predict()               predict()

    
    
    BaseWineModel
     │
     ├── defines WHAT
     │
     │   train()
     │   predict()
     │
     ↓
Child classes
     │
     ├── define HOW
     │
     │   Random Forest implementation
     │   Decision Tree implementation

     This is the relationship between abstraction, inheritance, and polymorphism.

In [82]:
#============================================================
# STEP 3
# Define the base model class
#============================================================

from abc import ABC, abstractmethod

class BaseWineModel(ABC):
    """
    OOP TOPIC: ABSTRACTION + INHERITANCE
    This abstract class defines WHAT every model must do:
        - train()
        - predict()
    Concrete models (RandomForest, XGBoost, etc.) will INHERIT from this class.
    """
#============================================================
# TRAINING AND PREDICTION
# OOP TOPIC: ABSTRACTION
# This class defines the interface for all wine models.
# Python gives us this functionality so that we can create a blueprint for other classes.
#============================================================
    @abstractmethod
    def train(self, X_train, y_train):
        pass

    @abstractmethod
    def predict(self, X_test):
        pass


In [83]:
#============================================================
# STEP 4
# Define a concrete model class (Random Forest)
#============================================================

from sklearn.ensemble import RandomForestClassifier

class RandomForestWineModel(BaseWineModel):
    """
    OOP TOPIC: INHERITANCE
    This class INHERITS from BaseWineModel.
    It must implement train() and predict().

    OOP TOPIC: POLYMORPHISM
    The pipeline can call:
        model.train(...)
        model.predict(...)
    without caring that this is a RandomForest model.
    """

    def __init__(self, n_estimators=100):# n_estimators is a parameter used in RandomForestClassifier. It tells the model how many decision trees to build inside the forest.
        self.n_estimators = n_estimators# store the number of trees in the forest as an instance variable
        self.model = None  # Will hold the trained RandomForest model

    # --------------------------------------------------------
    # Train the model
    # --------------------------------------------------------
    def train(self, X_train, y_train):

        # Create the Random Forest model.
        self.model = RandomForestClassifier(
            n_estimators=self.n_estimators,
            random_state=42
        )

        # Train the model.
        self.model.fit(X_train, y_train)

    # --------------------------------------------------------
    # Predict using the trained model
    # --------------------------------------------------------
    def predict(self, X_test):
        return self.model.predict(X_test)


In [84]:
# Decision Tree algorithm.
from sklearn.tree import DecisionTreeClassifier

class DecisionTreeWineModel(BaseWineModel):
    """
    OOP TOPIC: INHERITANCE
    This class INHERITS from BaseWineModel.
    It must implement train() and predict().

    OOP TOPIC: POLYMORPHISM
    The pipeline can call:
        model.train(...)
        model.predict(...)
    without caring that this is a DecisionTree model.
    """

    def __init__(self, max_depth=None):
        self.max_depth = max_depth # max_depth is a parameter used in DecisionTreeClassifier. It tells the model how deep the tree can grow. A deeper tree can capture more complex patterns but may overfit the training data.
        self.model = None  # Will hold the trained DecisionTree model

    # --------------------------------------------------------
    # Train the model
    # --------------------------------------------------------
    def train(self, X_train, y_train):

        # Create the Decision Tree model.
        self.model = DecisionTreeClassifier(
            max_depth=self.max_depth,
            random_state=42
        )

        # Train the model.
        self.model.fit(X_train, y_train)

    # --------------------------------------------------------
    # Predict using the trained model
    # --------------------------------------------------------
    def predict(self, X_test):
        return self.model.predict(X_test)

In [ ]:
# ============================================================
# MODULE: src/evaluation/evaluate.py
#
# PURPOSE:
#     Evaluate the performance of our machine-learning model
#     and generate evaluation artifacts.
#
# OOP CONCEPTS:
#
#     🟣 ENCAPSULATION
#         Hide internal artifact-saving operations inside
#         private methods.
#
#     🟡 COMPOSITION
#         WinePipeline will contain a ModelEvaluator object.
# ============================================================


# ============================================================
# IMPORT LIBRARIES
# ============================================================


# ------------------------------------------------------------
# Import matplotlib.
#
# We use matplotlib to create our confusion matrix plot.
# ------------------------------------------------------------

import matplotlib


# ------------------------------------------------------------
# Use the "Agg" backend.
#
# "Agg" allows matplotlib to create images without opening
# a graphical window.
#
# This is important when our application runs in:
#
#     Docker
#     Kubernetes
#     GitHub Actions
#     Linux servers
#
# These environments may not have a graphical interface.
# ------------------------------------------------------------

matplotlib.use("Agg")


# ------------------------------------------------------------
# Import pyplot.
#
# We use pyplot to create and save our chart.
# ------------------------------------------------------------

import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Import seaborn.
#
# Seaborn makes it easier to create a professional-looking
# confusion matrix heatmap.
# ------------------------------------------------------------

import seaborn as sns


# ------------------------------------------------------------
# Import the machine-learning evaluation metrics.
# ------------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# MODEL EVALUATOR CLASS
# ============================================================
#
# This class is responsible for evaluating model predictions
# and generating evaluation artifacts.
# ============================================================

class ModelEvaluator:

    # ========================================================
    # CONSTRUCTOR
    # ========================================================
    #
    # __init__() runs automatically when we create:
    #
    #     evaluator = ModelEvaluator()
    #
    # We use it to configure where our artifacts will be saved.
    # ========================================================

    def __init__(
        self,
        output_dir: str = "./artifacts"
    ):

        # ----------------------------------------------------
        # Store the main artifact directory inside the object.
        #
        # self.output_dir means:
        #
        # "This particular ModelEvaluator object owns this
        #  output directory."
        # ----------------------------------------------------

        self.output_dir = output_dir


    # ========================================================
    # 🟣 ENCAPSULATION
    #
    # PRIVATE METHOD:
    #
    # _save_classification_report()
    #
    # The underscore tells other developers:
    #
    # "This method is intended for internal use."
    #
    # The user of the class does not need to call this method
    # directly.
    # ========================================================

    def _save_classification_report(
        self,
        report_text: str
    ):

        # ----------------------------------------------------
        # Build the location where the report will be saved.
        #
        # Our final path will be:
        #
        # artifacts/reports/classification_report.txt
        # ----------------------------------------------------

        file_path = (
            f"{self.output_dir}/reports/"
            f"classification_report.txt"
        )


        # ----------------------------------------------------
        # Open the file in write mode.
        #
        # "w" means:
        #
        #     Create the file if it doesn't exist.
        #
        #     Replace the contents if it already exists.
        # ----------------------------------------------------

        with open(
            file_path,
            "w"
        ) as file:

            # ------------------------------------------------
            # Write the classification report into the file.
            # ------------------------------------------------

            file.write(
                report_text
            )


        # ----------------------------------------------------
        # Tell us where the artifact was saved.
        # ----------------------------------------------------

        print(
            f"[INFO] Classification report saved to "
            f"{file_path}"
        )


    # ========================================================
    # 🟣 ENCAPSULATION
    #
    # PRIVATE METHOD:
    #
    # _save_confusion_matrix_plot()
    #
    # This method hides all the matplotlib/seaborn details
    # from the rest of our application.
    # ========================================================

    def _save_confusion_matrix_plot(
        self,
        cm
    ):

        # ----------------------------------------------------
        # Create a figure.
        # ----------------------------------------------------

        plt.figure(
            figsize=(6, 5)
        )


        # ----------------------------------------------------
        # Create a heatmap using the confusion matrix.
        #
        # annot=True
        #     Display the numbers inside the cells.
        #
        # fmt="d"
        #     Display values as integers.
        #
        # cmap="Blues"
        #     Use a blue color scale.
        # ----------------------------------------------------

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues"
        )


        # ----------------------------------------------------
        # Add a title to the chart.
        # ----------------------------------------------------

        plt.title(
            "Confusion Matrix"
        )


        # ----------------------------------------------------
        # Build the output path.
        #
        # Final location:
        #
        # artifacts/plots/confusion_matrix.png
        # ----------------------------------------------------

        file_path = (
            f"{self.output_dir}/plots/"
            f"confusion_matrix.png"
        )


        # ----------------------------------------------------
        # Save the chart as a PNG image.
        # ----------------------------------------------------

        plt.savefig(
            file_path,
            bbox_inches="tight"
        )


        # ----------------------------------------------------
        # Close the figure.
        #
        # This is important because it releases matplotlib
        # resources.
        # ----------------------------------------------------

        plt.close()


        # ----------------------------------------------------
        # Display an informational message.
        # ----------------------------------------------------

        print(
            f"[INFO] Confusion matrix plot saved to "
            f"{file_path}"
        )


    # ========================================================
    # PUBLIC METHOD
    #
    # evaluate()
    #
    # This is the method that the rest of our application
    # will normally call.
    #
    # For example:
    #
    #     evaluator.evaluate(
    #         y_test,
    #         predictions
    #     )
    # ========================================================

    def evaluate(
        self,
        y_test,
        predictions
    ) -> dict:

        # ----------------------------------------------------
        # Tell the user that evaluation has started.
        # ----------------------------------------------------

        print(
            "[INFO] Evaluating model performance..."
        )


        # ====================================================
        # STEP 1 — CALCULATE ACCURACY
        # ====================================================

        # ----------------------------------------------------
        # Compare actual values against predictions.
        # ----------------------------------------------------

        accuracy = accuracy_score(
            y_test,
            predictions
        )


        # ====================================================
        # STEP 2 — GENERATE CLASSIFICATION REPORT
        # ====================================================

        # ----------------------------------------------------
        # Generate the text report containing:
        #
        #     precision
        #     recall
        #     f1-score
        #     support
        # ----------------------------------------------------

        report = classification_report(
            y_test,
            predictions
        )


        # ====================================================
        # STEP 3 — GENERATE CONFUSION MATRIX
        # ====================================================

        # ----------------------------------------------------
        # Calculate the confusion matrix.
        # ----------------------------------------------------

        cm = confusion_matrix(
            y_test,
            predictions
        )


        # ====================================================
        # STEP 4 — SAVE ARTIFACTS
        #
        # 🟣 ENCAPSULATION
        #
        # We don't put file-writing and plotting logic here.
        #
        # Instead, we call our private methods.
        # ====================================================

        self._save_classification_report(
            report
        )


        self._save_confusion_matrix_plot(
            cm
        )


        # ====================================================
        # STEP 5 — RETURN RESULTS
        # ====================================================

        # ----------------------------------------------------
        # Return all evaluation information in a dictionary.
        # ----------------------------------------------------

        return {

            # Model accuracy.
            "accuracy": accuracy,

            # Text classification report.
            "classification_report": report,

            # Numerical confusion matrix.
            "confusion_matrix": cm
        }